# Drop Columns + Rerun Baseline

Name: Ethan He, Rhett Duan
NetID: ehe5, rhettd2
Ticket: dp5
GitHub Issue: #35

Goal: Remove columns marked “yes leakage” and compare metrics.

In [13]:

import pandas as pd
import numpy as np


df = pd.read_csv("../data/raw/stroke_data.csv")
df.head()


,stroke,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,0,2,2,5,1,0,0,2,2,3,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,0,2,2,1,1,0,0,1,2,3,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,1,2,3,1,1,1,2,1,3,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233
3,0,2,3,3,1,1,1,2,1,4,...,1605,70.99,143.37,10.0,81.60,24.527,30.567,18.174,1766,3706
4,0,1,1,4,1,0,0,2,1,2,...,1818,74.75,229.45,14.2,67.49,26.030,24.837,10.533,1842,2461


# Dropping Possible Leakage Columns
Yes Leakage: Stroke | General health condition | Minutes sedentary activity | Depression

In [14]:
df = df.drop(columns=["stroke", "General health condition", "depression", "Minutes sedentary activity"])

df.head()

,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,sleep time,diabetes,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,2,2,5,1,0,0,2,2,6.0,0,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,2,2,1,1,0,0,1,2,5.0,0,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,2,3,1,1,1,2,1,8.0,0,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233
3,2,3,3,1,1,1,2,1,6.0,1,...,1605,70.99,143.37,10.0,81.60,24.527,30.567,18.174,1766,3706
4,1,1,4,1,0,0,2,1,4.0,0,...,1818,74.75,229.45,14.2,67.49,26.030,24.837,10.533,1842,2461


# Rerunning Baseline

In [15]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Try "balanced" later to address class imbalance
CLASS_WEIGHT = None

# Logistic regression settings
MAX_ITER = 1000
SOLVER = "liblinear"

import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "stroke_data.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.head()

if "stroke" not in df.columns:
    raise ValueError("Dataset must contain a 'stroke' column.")

y = df["stroke"]
X = df.drop(columns=["stroke", "General health condition", "depression", "Minutes sedentary activity"])


print("Label counts")
print(y.value_counts(dropna=False))
print("\nLabel proportions")
print(y.value_counts(normalize=True, dropna=False))

try:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    stratify_used = "yes"
    stratify_note = ""
except ValueError as e:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=None,
    )
    stratify_used = "no"
    stratify_note = str(e)

print(f"Split standardized: test_size={TEST_SIZE}, seed={RANDOM_STATE}, stratify={stratify_used}")
if stratify_note:
    print("Stratify note")
    print(stratify_note)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Number of categorical columns", len(cat_cols))
print("Number of numeric columns", len(num_cols))

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

model = LogisticRegression(
    max_iter=MAX_ITER,
    solver=SOLVER,
    class_weight=CLASS_WEIGHT,
    random_state=RANDOM_STATE,
)

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", model),
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

try:
    roc_auc = roc_auc_score(y_test, y_prob)
except Exception:
    roc_auc = None

cm = confusion_matrix(y_test, y_pred)

print("Accuracy", accuracy)
print("Precision", precision)
print("Recall", recall)
print("ROC AUC", roc_auc)
print("Confusion matrix")
print(cm)

Label counts
stroke
0    4241
1     362
Name: count, dtype: int64

Label proportions
stroke
0    0.921356
1    0.078644
Name: proportion, dtype: float64
Split standardized: test_size=0.2, seed=42, stratify=yes
Number of categorical columns 0
Number of numeric columns 32
Accuracy 0.9218241042345277
Precision 0.0
Recall 0.0
ROC AUC 0.5966332940714567
Confusion matrix
[[849   0]
 [ 72   0]]


# Saving Report to baseline_metric.md
ROC AUC changed from the old version



In [16]:
reports_dir = ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / "baseline_metrics.md" # Change this to a different name if you are doing testing and don't want to change the baseline model

with open(report_path, "w", encoding="utf-8") as f:
    f.write("# Baseline Metrics\n\n")
    f.write("## Model\n")
    f.write("Logistic Regression\n\n")

    f.write("## Split\n")
    f.write(f"test_size: {TEST_SIZE}\n")
    f.write(f"random_state: {RANDOM_STATE}\n")
    f.write(f"stratify: {stratify_used}\n")
    if stratify_note:
        f.write("stratify_note:\n")
        f.write(f"{stratify_note}\n")
    f.write("\n")

    f.write("## Label balance\n")
    f.write("Counts\n")
    f.write(y.value_counts(dropna=False).to_string())
    f.write("\n\n")

    f.write("## Metrics on test set\n")
    f.write(f"Accuracy: {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")
    f.write(f"ROC AUC: {roc_auc if roc_auc is not None else 'N A'}\n\n")

    f.write("## Confusion matrix on test set\n")
    f.write("Format is [[TN FP]\n")
    f.write("           [FN TP]]\n\n")
    f.write(str(cm))
    f.write("\n")

print(f"Saved report to {report_path}")
print(
    f"Baseline standardized: test_size={TEST_SIZE}, seed={RANDOM_STATE}, stratify={stratify_used}, metrics saved in reports/baseline_metrics.md"
)


Saved report to /Users/ethanhe/Desktop/UIUC/HAS/stroke-prevention-demo/reports/baseline_metrics.md
Baseline standardized: test_size=0.2, seed=42, stratify=yes, metrics saved in reports/baseline_metrics.md
